<a href="https://colab.research.google.com/github/ErnestLinK/Enigma-Share/blob/main/RFMO_IUU_Web_Scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Download and install packages
!pip install pdfplumber
!pip install BeautifulSoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 95.8 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [ ]:
#Import packages for scraping
import pandas as pd
import pdfplumber
import requests
from io import BytesIO, StringIO
from bs4 import BeautifulSoup
import re
import json


In [ ]:
# Initialise
# Create empty DataFrame with specified columns as base
df_iuu = pd.DataFrame(columns=[
    'Vessel Name',
    'IMO Number',
    'International Radio Call Sign',
    'Flag State',
    'Gear Type',
    'Vessel Length (LOA, in meters)',
    'Vessel Tonnage (GT)',
    'Vessel Depth/Draft',
    'Owner Name',
    'Owner Address',
    'Owner Place Registration',
    'Operator Name',
    'Operator Address',
    'Year Built',
    'Activity Date',
    'Activity Description',
    'Additional Information/Notes',
    'Date Listed From',
    'RFMO',
    'Source RFMO',
    'Source File Type',
    'Source URL',
    'Last Checked',
    'IUU Number',
    'Action Taken',
    'Outcome',
    'Previous Names',
    'Previous Call Signs',
    'Previous Flag States',
    'Previous Operators',
    'Previous Owners',
    'Cross Listings'
])

In [ ]:
# ICCAT

# Read two excel sheets by name
df_IUUs = pd.read_excel('https://www.iccat.int/Data/IUU/IUU.xlsx', sheet_name='IUUs')
df_HstIUUS = pd.read_excel('https://www.iccat.int/Data/IUU/IUU.xlsx', sheet_name='HstIUUs')

# Create a new DataFrame with the correct mapping
df_map_IUUs = pd.DataFrame({
    'Vessel Name': df_IUUs['VesselName'],
    'IMO Number': df_IUUs['IMONo'],
    'International Radio Call Sign': df_IUUs['IRCS'],
    'Flag State': df_IUUs['FlagIUU'],
    'Gear Type': df_IUUs['Gear'],
    'Vessel Length (LOA, in meters)': df_IUUs['LOAm'],
    'Owner Name': df_IUUs['OwnerName'],
    'Owner Address': df_IUUs['OwnerAddress'],
    'Owner Place Registration': df_IUUs['OwnerPlaceRegistration'],
    'Operator Name': df_IUUs['OperatorName'],
    'Operator Address': df_IUUs['OperatorAddress'],
    'Year Built': df_IUUs['DocDate'],
    'Activity Date': df_IUUs['ActvDate'],
    'Activity Description': df_IUUs['Activity'],
    'Additional Information/Notes': df_IUUs['AddInfo'],
    'Date Listed From': df_IUUs['DateFrom'],
    'RFMO': "ICCAT",
    'Source RFMO': df_IUUs['RepRFMO'],
    'Source File Type': "XLSX",
    'Source URL': "https://www.iccat.int/Data/IUU/IUU.xlsx",
    'IUU Number':df_IUUs['VesIUUID'],
    'Action Taken': df_IUUs['ActvActions'],
    'Outcome':df_IUUs['ActvOutcome']
})

df_iuu = pd.concat([df_iuu, df_map_IUUs], ignore_index=True)

df_map_HstIUUs = pd.DataFrame({
    'Vessel Name': df_HstIUUS['VesselName'],
    'IMO Number': df_HstIUUS['IMONo'],
    'International Radio Call Sign': df_HstIUUS['IRCS'],
    'Flag State': df_HstIUUS['FlagIUU'],
    'Gear Type': df_HstIUUS['Gear'],
    'Vessel Length (LOA, in meters)': df_HstIUUS['LOAm'],
    'Owner Name': df_HstIUUS['OwnerName'],
    'Owner Address': df_HstIUUS['OwnerAddress'],
    'Owner Place Registration': df_HstIUUS['OwnerPlaceRegistration'],
    'Operator Name': df_HstIUUS['OperatorName'],
    'Operator Address': df_HstIUUS['OperatorAddress'],
    'Year Built': df_HstIUUS['DocDate'],
    'Activity Date': df_HstIUUS['ActvDate'],
    'Activity Description': df_HstIUUS['Activity'],
    'Additional Information/Notes': df_HstIUUS['AddInfo'],
    'Date Listed From': df_HstIUUS['DateFrom'],
    'RFMO': "ICCAT",
    'Source RFMO': df_HstIUUS['RepRFMO'],
    'Source File Type': "XLSX",
    'Source URL': "https://www.iccat.int/Data/IUU/IUU.xlsx",
    'IUU Number':df_HstIUUS['VesIUUID'],
    'Action Taken': df_HstIUUS['ActvActions'],
    'Outcome':df_HstIUUS['ActvOutcome']
})

df_iuu = pd.concat([df_iuu, df_map_HstIUUs], ignore_index=True)
print(f"Number of records: {len(df_iuu)}")

# Export to CSV
#df.to_csv('RFMO_IUU.csv', index=False)

# Download the file
#files.download('RFMO_IUU.csv')

Number of records: 393


In [ ]:
# IOTC

# PDF URL
pdf_url = "https://iotc.org/sites/default/files/documents/compliance/vessel_lists/IUU%20lists/IOTC_IUU_Vessels_List_20250416EF.pdf"

# Download and read PDF
response = requests.get(pdf_url)
pdf_file = BytesIO(response.content)

# Extract tables from all pages
all_tables = []
with pdfplumber.open(pdf_file) as pdf:
    for page_num, page in enumerate(pdf.pages):
        print(f"Processing page {page_num + 1}...")
        table = page.extract_table()
        if table:
            all_tables.append(table)

# Combine all tables
if all_tables:
    # Use first table's first row as headers
    headers = all_tables[0][0]
    all_data = []

    # Add data from all tables (skip headers for subsequent tables)
    for i, table in enumerate(all_tables):
        if i == 0:
            all_data.extend(table[1:])  # Skip header for first table
        else:
            # Check if first row is header, if so skip it
            if table[0] == headers:
                all_data.extend(table[1:])
            else:
                all_data.extend(table)

    # Create DataFrame
    df_iotc = pd.DataFrame(all_data, columns=headers)
    df_iotc.columns.values[0] = 'IOTC IUU No'
    df_iotc.columns.values[1] = 'Current name of vessel (previous names)'
    df_iotc.columns.values[2] = 'Current flag (previous flags)'
    df_iotc.columns.values[3] = 'Owner / beneficial owners (previous owners)'
    df_iotc.columns.values[4] = 'Owner Address (Previous Owner Addresses)'
    df_iotc.columns.values[5] = 'Operator (previous operators)'
    df_iotc.columns.values[6] = 'Operator Address (Previous Operator Addresses)'
    df_iotc.columns.values[7] = 'Country and number of registration of legal entity'
    df_iotc.columns.values[8] = 'Lloyds-IMO number or unique vessel identifier'
    df_iotc.columns.values[9] = 'Call sign (previous call signs)'
    df_iotc.columns.values[10] = 'Recent photographs of the vessel'
    df_iotc.columns.values[11] = 'Length Overall (LOA) of the vessel(m)'
    df_iotc.columns.values[12] = 'Date first included in IOTC IUU List'
    df_iotc.columns.values[13] = 'Summary of IUU activities'
    df_iotc.columns.values[14] = 'Reference of supporting document and information'
    df_iotc.columns.values[15] = 'Summary of action taken'
    df_iotc.columns.values[16] = 'Source organisation'

    #df_iotc['Current_Name'] = df_iotc['Current name of vessel (previous names)'].str.split('(').str[0].str.strip()
    #df_iotc['Previous_Name'] = df_iotc['Current name of vessel (previous names)'].str.split('(').str[1].str.strip(')')

    # Create a new DataFrame with mapping
    df_map_iotc = pd.DataFrame({
    'Vessel Name': df_iotc['Current name of vessel (previous names)'],
    'IMO Number': df_iotc['Lloyds-IMO number or unique vessel identifier'],
    'International Radio Call Sign': df_iotc['Call sign (previous call signs)'],
    'Flag State': df_iotc['Current flag (previous flags)'],
    'Vessel Length (LOA, in meters)': df_iotc['Length Overall (LOA) of the vessel(m)'],
    'Owner Name': df_iotc['Owner / beneficial owners (previous owners)'],
    'Owner Address': df_iotc['Owner Address (Previous Owner Addresses)'],
    'Owner Place Registration': df_iotc['Country and number of registration of legal entity'],
    'Operator Name': df_iotc['Operator (previous operators)'],
    'Operator Address': df_iotc['Operator Address (Previous Operator Addresses)'],
    'Activity Description': df_iotc['Summary of IUU activities'],
    'Additional Information/Notes': df_iotc['Reference of supporting document and information'],
    'Date Listed From': df_iotc['Date first included in IOTC IUU List'],
    'RFMO': "IOTC",
    'Source RFMO': df_iotc['Source organisation'],
    'Source File Type': "PDF",
    'Source URL': "https://iotc.org/sites/default/files/documents/compliance/vessel_lists/IUU%20lists/IOTC_IUU_Vessels_List_20250416EF.pdf",
    'IUU Number':df_iotc['IOTC IUU No'],
    'Action Taken': df_iotc['Summary of action taken']
})

    print("New columns:", df_iotc.columns.tolist())
    #df_iotc.to_csv("output.csv", index=False)

    df_iuu = pd.concat([df_iuu, df_map_iotc], ignore_index=True)
    print(f"Number of records: {len(df_iuu)}")

    #df.to_csv("output.csv", index=False)

    #print(f"Extracted {len(all_data)} rows from {len(all_tables)} pages")
    #files.download("output.csv")
else:
    print("No tables found in PDF")

Processing page 1...
Processing page 2...
Processing page 3...
Processing page 4...
Processing page 5...
Processing page 6...
Processing page 7...
Processing page 8...
Processing page 9...
Processing page 10...
Processing page 11...
Processing page 12...
Processing page 13...
New columns: ['IOTC IUU No', 'Current name of vessel (previous names)', 'Current flag (previous flags)', 'Owner / beneficial owners (previous owners)', 'Owner Address (Previous Owner Addresses)', 'Operator (previous operators)', 'Operator Address (Previous Operator Addresses)', 'Country and number of registration of legal entity', 'Lloyds-IMO number or unique vessel identifier', 'Call sign (previous call signs)', 'Recent photographs of the vessel', 'Length Overall (LOA) of the vessel(m)', 'Date first included in IOTC IUU List', 'Summary of IUU activities', 'Reference of supporting document and information', 'Summary of action taken', 'Source organisation']
Number of records: 599


In [ ]:
# CCSBT

pdf_url = "https://www.ccsbt.org/sites/default/files/userfiles/file/docs_english/CCSBT%20IUU%20List.pdf"

# Download and read PDF
response = requests.get(pdf_url)
pdf_file = BytesIO(response.content)

# Extract tables from all pages
all_tables = []
with pdfplumber.open(pdf_file) as pdf:
    for page_num, page in enumerate(pdf.pages):
        print(f"Processing page {page_num + 1}...")
        table = page.extract_table()
        if table:
            all_tables.append(table)

# Combine all tables
if all_tables:
    # Use first table's first row as headers
    headers = all_tables[0][0]
    all_data = []

    # Add data from all tables (skip headers for subsequent tables)
    for i, table in enumerate(all_tables):
        if i == 0:
            all_data.extend(table[1:])  # Skip header for first table
        else:
            # Check if first row is header, if so skip it
            if table[0] == headers:
                all_data.extend(table[1:])
            else:
                all_data.extend(table)

    # Create DataFrame
    df_ccsbt = pd.DataFrame(all_data, columns=headers)
    print("New columns:", df_ccsbt.columns.tolist())

    #df_ccsbt = df_ccsbt.drop(columns=[df_ccsbt.columns[6], df_ccsbt.columns[7]])

    df_ccsbt.columns = [
    'vessel_name',
    'flag',
    'owner',
    'operator',
    'call_sign',
    'imo_number',
    'drop1',
    'drop2',
    'photos_available',
    'date_first_included',
    'iuu_activities',
    'source_organisation',
    'date_original_listing',
    'drop3',
    'drop4'
    ]

    # Drop the empty columns
    df_ccsbt = df_ccsbt.drop(columns=['drop1', 'drop2', 'drop3', 'drop4'])


    #df_ccsbt.to_csv("df_ccsbt_raw.csv", index=False)
    #files.download("df_ccsbt_raw.csv")

    '''
    df_ccsbt.columns.values[0] = 'Current Name of Vessel (and any previous names)'
    df_ccsbt.columns.values[1] = 'Current Flag of Vessel (and any previous flags)'
    df_ccsbt.columns.values[2] = 'Owner/beneficial owner (and any previous owners)'
    df_ccsbt.columns.values[3] = 'Operator (and any previous operators)'
    df_ccsbt.columns.values[4] = 'Call Sign (and any previous call signs)'
    df_ccsbt.columns.values[5] = 'Lloyds/IMO/UVI Number (Registration Number)'
    df_ccsbt.columns.values[6] = 'Photographs Available'
    df_ccsbt.columns.values[7] = 'Date First Included on CCSBT IUU Vessel List'
    df_ccsbt.columns.values[8] = 'Summary of IUU Activities'
    df_ccsbt.columns.values[9] = 'Original Source Listing Organisation'
    df_ccsbt.columns.values[10] = 'Date/Year Vessel 1st Included on the Original Source Listing Organisation IUU List'
    '''

    #df_ccsbt = df_ccsbt.dropna(subset=['Current Name of Vessel (and any previous names)'])
    df_ccsbt = df_ccsbt.dropna(how='all')

    # Create a new DataFrame with mapping
    df_map_ccsbt = pd.DataFrame({
    'Vessel Name': df_ccsbt['vessel_name'],
    'IMO Number': df_ccsbt['imo_number'],
    'International Radio Call Sign': df_ccsbt['call_sign'],
    'Flag State': df_ccsbt['flag'],
    'Owner Name': df_ccsbt['owner'],
    'Operator Name': df_ccsbt['operator'],
    'Activity Description': df_ccsbt['iuu_activities'],
    'Date Listed From': df_ccsbt['date_original_listing'],
    'RFMO': "CCSBT",
    'Source RFMO': df_ccsbt['source_organisation'],
    'Source File Type': "PDF",
    'Source URL': pdf_url
})

    df_iuu = pd.concat([df_iuu, df_map_ccsbt], ignore_index=True)
    print(f"Number of records: {len(df_iuu)}")


    #print(f"Extracted {len(all_data)} rows from {len(all_tables)} pages")
    #df_ccsbt.to_csv("df_ccsbt.csv", index=False)
    #files.download("df_ccsbt.csv")


Processing page 1...
Processing page 2...
Processing page 3...
Processing page 4...
Processing page 5...
Processing page 6...
Processing page 7...
Processing page 8...
Processing page 9...
Processing page 10...
Processing page 11...
Processing page 12...
Processing page 13...
Processing page 14...
Processing page 15...
Processing page 16...
Processing page 17...
Processing page 18...
Processing page 19...
Processing page 20...
Processing page 21...
Processing page 22...
New columns: ['Current\nName of\nVessel\n(and any\nprevious\nnames)', 'Current\nFlag of\nVessel\n(and any\nprevious\nflags)', 'Owner/ beneficial\nowner\n(and any previous\nowners)', 'Operator\n(and any\nprevious\noperators)', 'Call Sign\n(and any\nprevious\ncall\nsigns)', '', 'Lloyds/', '', 'Photo-\ngraphs\nAvail-\nable?', 'Date First\nIncluded\non CCSBT\nIUU\nVessel\nList', 'Summary of IUU Activities\n(including date and location of\nany relevant sightings)', 'Original\nSource\nListing\nOrganisa-\ntion', '', 'Date/ Yea

In [ ]:
"""
WCPFC IUU Vessel List Extractor

This script extracts the Western and Central Pacific Fisheries Commission (WCPFC)
Illegal, Unreported and Unregulated (IUU) vessel list data from their API.

https://iuu.imcsnet.org/static/iuu-vessel-list.js fetches data from an API endpoint and the IUU vessel data dynamically.
"""

import requests
import json
import pandas as pd
from datetime import datetime
import os
import sys
from typing import Dict, List, Any, Optional

class WCPFCIUUExtractor:
    """Extract and process WCPFC IUU vessel data"""

    def __init__(self):
        self.base_url = "https://iuu.imcsnet.org"
        self.rfmo_id = "3"  # WCPFC RFMO ID
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Accept-Language': 'en-US,en;q=0.9',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
        }
        self.session = requests.Session()
        self.session.headers.update(self.headers)

    def extract_vessel_data(self) -> Optional[Dict[str, Any]]:
        """Extract raw vessel data from the WCPFC IUU API"""
        api_url = f"{self.base_url}/rfmo/{self.rfmo_id}/final"

        try:
            print(f"Fetching data from: {api_url}")
            response = self.session.get(api_url, timeout=30)
            response.raise_for_status()

            data = response.json()
            print(f"Successfully retrieved data for {len(data)} vessels")
            return data

        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            return None
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON response: {e}")
            return None

    def get_field_value(self, field_data: Any) -> str:
        """Extract value from field data structure"""
        if not field_data:
            return ""

        if isinstance(field_data, list) and len(field_data) > 0:
            if isinstance(field_data[0], dict) and 'value' in field_data[0]:
                return str(field_data[0]['value']) if field_data[0]['value'] is not None else ""

        return str(field_data) if field_data is not None else ""

    def get_array_values(self, field_data: Any, field_name: str = 'value') -> List[str]:
        """Extract array of values from field data"""
        if not field_data or not isinstance(field_data, list):
            return []

        values = []
        for item in field_data:
            if isinstance(item, dict) and field_name in item and item[field_name]:
                values.append(str(item[field_name]))

        return values

    def get_processed_value(self, field_data: Any) -> str:
        """Extract processed HTML content, fallback to regular value"""
        if not field_data:
            return ""

        if isinstance(field_data, list) and len(field_data) > 0:
            item = field_data[0]
            if isinstance(item, dict):
                # Try processed field first (HTML content)
                if 'processed' in item and item['processed']:
                    # Remove HTML tags for clean text
                    import re
                    clean_text = re.sub(r'<[^>]+>', '', item['processed'])
                    return clean_text.strip()
                # Fallback to value field
                elif 'value' in item and item['value']:
                    return str(item['value'])

        return ""

    def extract_company_info(self, company_data: List[Dict]) -> Dict[str, str]:
        """Extract company information (owner, operator, etc.)"""
        if not company_data or not isinstance(company_data, list) or len(company_data) == 0:
            return {
                'name': '',
                'address': '',
                'country': '',
                'imo': '',
                'reg_no': ''
            }

        company = company_data[0]
        return {
            'name': company.get('name', '') or '',
            'address': company.get('address', '') or '',
            'country': company.get('reg_country_iso2', '') or '',
            'imo': company.get('company_imo', '') or '',
            'reg_no': company.get('company_reg_no', '') or ''
        }

    def extract_master_info(self, master_data: List[Dict]) -> Dict[str, str]:
        """Extract vessel master information"""
        if not master_data or not isinstance(master_data, list) or len(master_data) == 0:
            return {'name': '', 'nationality': ''}

        master = master_data[0]
        return {
            'name': master.get('name', '') or '',
            'nationality': master.get('nationality_iso2', '') or ''
        }

    def format_date(self, date_string: str) -> str:
        """Format date string to readable format"""
        if not date_string:
            return ""

        try:
            # Handle ISO date format
            if 'T' in date_string:
                date_part = date_string.split('T')[0]
            else:
                date_part = date_string

            # Parse and reformat
            date_obj = datetime.strptime(date_part, '%Y-%m-%d')
            return date_obj.strftime('%d-%b-%Y')
        except:
            return date_string

    def process_vessel_data(self, raw_data: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Process raw vessel data into structured format"""
        if not raw_data:
            return []

        vessels = []

        for vessel_id, vessel_data in raw_data.items():
            # Extract basic vessel information
            vessel_info = {
                'vessel_id': self.get_field_value(vessel_data.get('iuu_id')),
                'vessel_name': self.get_field_value(vessel_data.get('iuu_vessel_name')),
                'previous_names': ', '.join(self.get_array_values(vessel_data.get('iuu_prev_vessel_name', []))),

                # Identification
                'call_sign': self.get_field_value(vessel_data.get('iuu_ircs')),
                'previous_call_signs': ', '.join(self.get_array_values(vessel_data.get('iuu_prev_ircs', []))),
                'imo_number': self.get_field_value(vessel_data.get('iuu_imo')),
                'mmsi': self.get_field_value(vessel_data.get('iuu_mmsi')),
                'previous_mmsi': ', '.join(self.get_array_values(vessel_data.get('iuu_prev_mmsi', []))),
                'registration_number': self.get_field_value(vessel_data.get('iuu_reg_number')),
                'previous_reg_numbers': ', '.join(self.get_array_values(vessel_data.get('iuu_prev_reg_number', []))),

                # Flag information
                'flag_state': self.get_field_value(vessel_data.get('iuu_flag_iso2')),
                'previous_flag_states': ', '.join(self.get_array_values(vessel_data.get('iuu_prev_flag_iso2', []))),

                # Technical specifications
                'length': self.get_field_value(vessel_data.get('iuu_length')),
                'length_unit': self.get_field_value(vessel_data.get('iuu_length_unit')),
                'length_type': self.get_field_value(vessel_data.get('iuu_length_type')),
                'depth': self.get_field_value(vessel_data.get('iuu_depth')),
                'depth_unit': self.get_field_value(vessel_data.get('iuu_depth_unit')),
                'gross_tonnage': self.get_field_value(vessel_data.get('iuu_gross_tonnage')),
                'year_built': self.get_field_value(vessel_data.get('iuu_year_built')),
                'built_in': self.get_field_value(vessel_data.get('iuu_built_in')),
                'vessel_type': vessel_data.get('iuu_vessel_type', ''),

                # Listing information
                'listing_date': self.format_date(self.get_field_value(vessel_data.get('iuu_listing_effective_from'))),
                'delisting_date': self.format_date(self.get_field_value(vessel_data.get('iuu_delisting_effective_from'))),
                'delisting_reason': self.get_field_value(vessel_data.get('iuu_delisting_reason')),
                'notifying_member': self.get_field_value(vessel_data.get('iuu_notifying_rfmo_member')),

                # IUU Activity details
                'iuu_activity_summary': self.get_processed_value(vessel_data.get('iuu_act_summary')),
                'activity_position': self.get_processed_value(vessel_data.get('iuu_act_position')),
                'result_action': self.get_processed_value(vessel_data.get('iuu_act_result_action')),
                'sighting_details': self.get_processed_value(vessel_data.get('iuu_act_sighting_detail')),
                'violation_reference': ', '.join(self.get_array_values(vessel_data.get('iuu_act_violation_ref', []))),

                # Additional information
                'last_known_situation': self.get_processed_value(vessel_data.get('iuu_last_known_situation')),
                'other_notes': self.get_processed_value(vessel_data.get('iuu_other_notes')),
                'keywords': self.get_field_value(vessel_data.get('iuu_keywords')),
                'also_listed_by': vessel_data.get('iuu_also_listed_by_rfmos', ''),

                # Cross-listing information
                'cross_list_sources': ', '.join([
                    item.get('rfmo', '') for item in vessel_data.get('iuu_cross_list_source', [])
                    if isinstance(item, dict) and item.get('rfmo')
                ]),

                # RFMO vessel IDs
                'rfmo_vessel_ids': ', '.join([
                    f"{item.get('rfmo', '')}: {item.get('vessel_id', '')}"
                    for item in vessel_data.get('iuu_rfmo_vid', [])
                    if isinstance(item, dict) and item.get('rfmo') and item.get('vessel_id')
                ]),

                # Timestamps
                'created_date': self.format_date(self.get_field_value(vessel_data.get('iuu_created'))),
                'last_modified': self.format_date(self.get_field_value(vessel_data.get('iuu_changed'))),
                'revision_timestamp': self.format_date(self.get_field_value(vessel_data.get('iuu_revision_timestamp'))),
            }

            # Extract master information
            master_info = self.extract_master_info(vessel_data.get('iuu_master', []))
            vessel_info.update({
                'master_name': master_info['name'],
                'master_nationality': master_info['nationality']
            })

            # Extract previous master information
            prev_masters = vessel_data.get('iuu_prev_master', [])
            if prev_masters:
                prev_master_names = [
                    master.get('name', '') for master in prev_masters
                    if isinstance(master, dict) and master.get('name')
                ]
                vessel_info['previous_masters'] = ', '.join(prev_master_names)
            else:
                vessel_info['previous_masters'] = ''

            # Extract owner information
            owner_info = self.extract_company_info(vessel_data.get('iuu_owner', []))
            vessel_info.update({
                'owner_name': owner_info['name'],
                'owner_address': owner_info['address'],
                'owner_country': owner_info['country'],
                'owner_imo': owner_info['imo'],
                'owner_reg_no': owner_info['reg_no']
            })

            # Extract beneficial owner information
            bene_owner_info = self.extract_company_info(vessel_data.get('iuu_bene_owner', []))
            vessel_info.update({
                'beneficial_owner_name': bene_owner_info['name'],
                'beneficial_owner_address': bene_owner_info['address'],
                'beneficial_owner_country': bene_owner_info['country'],
                'beneficial_owner_imo': bene_owner_info['imo'],
                'beneficial_owner_reg_no': bene_owner_info['reg_no']
            })

            # Extract operator information
            operator_info = self.extract_company_info(vessel_data.get('iuu_operator', []))
            vessel_info.update({
                'operator_name': operator_info['name'],
                'operator_address': operator_info['address'],
                'operator_country': operator_info['country'],
                'operator_imo': operator_info['imo'],
                'operator_reg_no': operator_info['reg_no']
            })

            vessels.append(vessel_info)

        return vessels

    def save_data(self, vessels: List[Dict[str, Any]], raw_data: Dict[str, Any]) -> None:
        """Save processed data to files"""
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

        # Save raw JSON data
        #raw_filename = f'wcpfc_iuu_vessels_raw_{timestamp}.json'
        #with open(raw_filename, 'w', encoding='utf-8') as f:
        #    json.dump(raw_data, f, indent=2, ensure_ascii=False)
        #print(f"Raw data saved to: {raw_filename}")

        # Save processed data as JSON
        #processed_filename = f'wcpfc_iuu_vessels_processed_{timestamp}.json'
        #with open(processed_filename, 'w', encoding='utf-8') as f:
        #    json.dump(vessels, f, indent=2, ensure_ascii=False)
        #print(f"Processed data saved to: {processed_filename}")

        # Save as CSV
        if vessels:
            df_wcpfc = pd.DataFrame(vessels)
            csv_filename = f'wcpfc_iuu_vessels_{timestamp}.csv'
            df_wcpfc.to_csv(csv_filename, index=False, encoding='utf-8')
            print(f"CSV data saved to: {csv_filename}")
            files.download(csv_filename)

            # Save summary statistics
            self.save_summary(df, timestamp)

        # Save latest versions (without timestamp)
        with open('wcpfc_iuu_vessels_latest.json', 'w', encoding='utf-8') as f:
            json.dump(vessels, f, indent=2, ensure_ascii=False)

        if vessels:
            df.to_csv('wcpfc_iuu_vessels_latest.csv', index=False, encoding='utf-8')

    def save_summary(self, df: pd.DataFrame, timestamp: str) -> None:
        """Generate and save summary statistics"""
        summary_filename = f'wcpfc_iuu_summary_{timestamp}.txt'

        with open(summary_filename, 'w', encoding='utf-8') as f:
            f.write("WCPFC IUU Vessel List Summary\n")
            f.write("=" * 50 + "\n\n")
            f.write(f"Extraction Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total Vessels: {len(df)}\n\n")

            # Flag state analysis
            if 'flag_state' in df.columns:
                flag_counts = df['flag_state'].value_counts()
                f.write("Vessels by Flag State:\n")
                for flag, count in flag_counts.items():
                    if flag:  # Only show non-empty flags
                        f.write(f"  {flag}: {count}\n")
                f.write("\n")

            # Previous flag analysis
            if 'previous_flag_states' in df.columns:
                prev_flags = df['previous_flag_states'].dropna()
                prev_flags = prev_flags[prev_flags != '']
                if len(prev_flags) > 0:
                    f.write(f"Vessels with Previous Flag States: {len(prev_flags)}\n\n")

            # Cross-listing analysis
            if 'also_listed_by' in df.columns:
                cross_listed = df['also_listed_by'].dropna()
                cross_listed = cross_listed[cross_listed != '']
                if len(cross_listed) > 0:
                    f.write("Cross-listed with Other RFMOs:\n")
                    for rfmo_list in cross_listed.value_counts().items():
                        f.write(f"  {rfmo_list[0]}: {rfmo_list[1]} vessels\n")
                    f.write("\n")

            # Listing date analysis
            if 'listing_date' in df.columns:
                listing_dates = df['listing_date'].dropna()
                listing_dates = listing_dates[listing_dates != '']
                if len(listing_dates) > 0:
                    f.write(f"Oldest Listing: {listing_dates.min()}\n")
                    f.write(f"Most Recent Listing: {listing_dates.max()}\n\n")

            # Vessel names
            if 'vessel_name' in df.columns:
                f.write("Vessel Names:\n")
                for name in df['vessel_name'].dropna():
                    if name:
                        f.write(f"  - {name}\n")

        print(f"Summary saved to: {summary_filename}")

    def run(self) -> bool:
        """Main execution method"""
        global df
        print("WCPFC IUU Vessel List Extractor")
        print("=" * 40)

        # Extract raw data
        raw_data = self.extract_vessel_data()
        if not raw_data:
            print("Failed to extract vessel data")
            return False

        # Process data
        print("Processing vessel data...")
        vessels = self.process_vessel_data(raw_data)
        df_wcpfc = pd.DataFrame(vessels)

        # Create a new DataFrame with mapping
        df_map_wcpfc = pd.DataFrame({
        'Vessel Name': df_wcpfc['vessel_name'],
        'IMO Number': df_wcpfc['imo_number'],
        'International Radio Call Sign': df_wcpfc['call_sign'],
        'Flag State': df_wcpfc['flag_state'],
        'Gear Type': df_wcpfc['vessel_type'],
        'Vessel Length (LOA, in meters)': df_wcpfc['length'],
        'Vessel Tonnage (GT)': df_wcpfc['gross_tonnage'],
        'Vessel Depth/Draft': df_wcpfc['depth'],
        'Owner Name': df_wcpfc['owner_name'],
        'Owner Address': df_wcpfc['owner_address'],
        'Owner Place Registration': df_wcpfc['owner_reg_no'],
        'Operator Name': df_wcpfc['operator_name'],
        'Operator Address': df_wcpfc['operator_address'],
        'Year Built': df_wcpfc['year_built'],
        'Activity Description': df_wcpfc['iuu_activity_summary'],
        'Additional Information/Notes': df_wcpfc['keywords'],
        'Date Listed From': df_wcpfc['listing_date'],
        'RFMO': "WCPFC",
        'Source RFMO': df_wcpfc['also_listed_by'],
        'Source File Type': "CSV",
        'Source URL': "https://www.wcpfc.int/vessels/iuu-vessel-list",
        'Last Checked': df_wcpfc['last_modified'],
        'IUU Number':df_wcpfc['vessel_id'],
        'Action Taken': df_wcpfc['result_action'],
        'Outcome': df_wcpfc['last_known_situation'],
        'Previous Call Signs': df_wcpfc['previous_call_signs'],
        'Previous Flag States': df_wcpfc['previous_flag_states']
        })
        global df_iuu
        df_iuu = pd.concat([df_iuu, df_map_wcpfc], ignore_index=True)
        print(f"Number of records: {len(df_iuu)}")

        if not vessels:
            print("No vessels found in the data")
            return False

        print(f"Processed {len(vessels)} vessels")

        # Save data
        #self.save_data(vessels, raw_data)

        # Display summary
        #print("\nSummary:")
        #print(f"Total vessels on IUU list: {len(vessels)}")

        # Show vessel names
        #vessel_names = [v['vessel_name'] for v in vessels if v['vessel_name']]
        #if vessel_names:
        #    print("Vessels:")
        #    for name in vessel_names:
        #        print(f"  - {name}")

        return True

def main():
    """Main function"""
    extractor = WCPFCIUUExtractor()

    try:
        success = extractor.run()
        if success:
            print("\nExtraction completed successfully!")
            #sys.exit(0)
        else:
            print("\nExtraction failed!")
            sys.exit(1)
    except KeyboardInterrupt:
        print("\nExtraction cancelled by user")
        sys.exit(1)
    except Exception as e:
        print(f"\nUnexpected error: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()

WCPFC IUU Vessel List Extractor
Fetching data from: https://iuu.imcsnet.org/rfmo/3/final
Successfully retrieved data for 4 vessels
Processing vessel data...
Number of records: 760
Processed 4 vessels

Extraction completed successfully!


In [ ]:
#IATTC

import requests
import pandas as pd

# Try the direct download approach
url = "https://www.iattc.org/en-US/Vessel/GetXLSIUUList"

try:
    # Add headers to mimic a browser request
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    response = requests.get(url, headers=headers)
    print(f"Status code: {response.status_code}")
    print(f"Content type: {response.headers.get('content-type', 'Unknown')}")

    if response.status_code == 200:
        # Save the file
        with open('iattc_iuu_list.xlsx', 'wb') as f:
            f.write(response.content)
        print("File downloaded successfully!")

        # Try to read the Excel file
        # Read all sheets from the Excel file
        excel_file = pd.ExcelFile('iattc_iuu_list.xlsx')

        print("Sheet names:")
        print(excel_file.sheet_names)
        print()

        # Check each sheet
        for sheet_name in excel_file.sheet_names:
            print(f"=== Sheet: {sheet_name} ===")
            df_iattc = pd.read_excel('iattc_iuu_list.xlsx', sheet_name=sheet_name)
            print(f"Shape: {df_iattc.shape}")
            print("First few rows:")
            print(df_iattc.head())
            print()

            # Alternative: Read with skiprows and define columns manually
            df_iattc = pd.read_excel('iattc_iuu_list.xlsx', sheet_name='IUUMasterList', skiprows=2)

            # Define proper column names based on what we saw
            column_names = ['IMO_Number', 'Vessel_Name', 'Flag', 'Gear', 'Previous_Name', 'Previous_Flag', 'Cross_Listings']
            df_iattc.columns = column_names
            # Start from row 2 (index 1) and drop empty rows
            df_iattc = df_iattc.iloc[1:].dropna(how='all')
            # Remove rows that are repeated headers
            df_iattc = df_iattc[~df_iattc.iloc[:, 0].str.contains('IMO number|IATTC adopted|IUU Vessels listed by other RFMOs', na=False)]
            # Drop the last 2 rows. Special Note rows.
            df_iattc = df_iattc.iloc[:-2]

            # Create a new DataFrame with mapping
            df_map_iattc = pd.DataFrame({
            'Vessel Name': df_iattc['Vessel_Name'],
            'IMO Number': df_iattc['IMO_Number'],
            'International Radio Call Sign': "",
            'Flag State': df_iattc['Flag'],
            'Gear Type': df_iattc['Gear'],
            'RFMO': "IATTC",
            'Source File Type': "XLSX",
            'Source URL': "https://www.iattc.org/en-US/Vessel/GetXLSIUUList",
            'Previous Names': df_iattc['Previous_Name'],
            'Previous Flag States': df_iattc['Previous_Flag'],
            'Cross Listings': df_iattc['Cross_Listings']
            })

            df_iuu = pd.concat([df_iuu, df_map_iattc], ignore_index=True)
            print(f"Number of records: {len(df_iuu)}")


            # Save the cleaned data
            #df_iattc.to_csv('iattc_iuu_vessels.csv', index=False)
            #print("\nCleaned data saved to 'iattc_iuu_vessels.csv'")
            #files.download("iattc_iuu_vessels.csv")
            #df_iuu.to_csv('df_iuu.csv', index=False)
            #files.download("df_iuu.csv")

    else:
        print(f"Failed to download. Status: {response.status_code}")
        print(f"Response content: {response.text[:500]}")

except Exception as e:
    print(f"Request failed: {e}")



Status code: 200
Content type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
File downloaded successfully!
Sheet names:
['Info', 'IUUMasterList']

=== Sheet: Info ===
Shape: (1, 1)
First few rows:
       Creation date: 27 Sep 2026 11:16 PM
0  Inter-American Tropical Tuna Commission

Number of records: 963
=== Sheet: IUUMasterList ===
Shape: (213, 7)
First few rows:
  IATTC adopted list       Unnamed: 1 All IUU Resolutions Unnamed: 3  \
0         IMO number           Vessel                Flag       Gear   
1                  -  Bhaskara No. 10                 UNK         LL   
2                  -   Bhaskara No. 9                 UNK         LL   
3                  -          Camelot                 UNK         LL   
4                  -       Dragon III                 UNK         LL   

      Unnamed: 4     Unnamed: 5                                    Unnamed: 6  
0  Previous name  Previous flag                                Cross listings  
1            NaN   

In [ ]:
#GFCM
#Downloaded and uploaded to Google Drive. Sharing publicly since this is public data.

import requests
import pdfplumber
import io
import pandas as pd


def process_pdf():
    # Download PDF
    #url = f"https://drive.google.com/uc?export=download&id={file_id}"
    url = f"https://raw.githubusercontent.com/ErnestLinK/Enigma-Share/main/GFCM-IUUVesselList-2025-November.pdf"
    response = requests.get(url)

    all_dfs = []

    with pdfplumber.open(io.BytesIO(response.content)) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            tables = page.extract_tables()

            for table_num, table in enumerate(tables, 1):
                if not table:
                    continue

                df = pd.DataFrame(table)

                # Remove first row containing 'VESSEL'
                vessel_mask = df.astype(str).apply(lambda row: row.str.fullmatch('VESSEL', case=False, na=False).any(), axis=1)
                if vessel_mask.any():
                    first_vessel_idx = vessel_mask.idxmax()
                    df = df.drop(first_vessel_idx).reset_index(drop=True)

                # Drop null rows
                df = df.dropna(how='all').reset_index(drop=True)

                # Skip if empty or less than 2 columns
                if df.empty or len(df.columns) < 2:
                    continue

                # Clean first column (will become column headers)
                first_col = df.iloc[:, 0].fillna('Unknown').astype(str).str.strip()

                # Get data columns (will become rows)
                data_cols = df.iloc[:, 1:]

                # Transpose: first column becomes headers, data columns become rows
                pivoted_df = data_cols.T
                pivoted_df.columns = first_col

                # Make column names unique
                cols = pivoted_df.columns.tolist()
                seen = {}
                unique_cols = []
                for col in cols:
                    if col in seen:
                        seen[col] += 1
                        unique_cols.append(f"{col}_{seen[col]}")
                    else:
                        seen[col] = 0
                        unique_cols.append(col)

                pivoted_df.columns = unique_cols

                all_dfs.append(pivoted_df)

    # Combine all dataframes horizontally
    if all_dfs:
        # Get all unique column names
        all_columns = set()
        for df in all_dfs:
            all_columns.update(df.columns)

        # Reindex all dataframes to have same columns
        aligned_dfs = []
        for df in all_dfs:
            aligned_df = df.reindex(columns=sorted(all_columns))
            aligned_dfs.append(aligned_df)

        # Concatenate vertically (each table becomes additional rows)
        combined = pd.concat(aligned_dfs, axis=0, ignore_index=True)


        # Drop specific rows
        rows_to_drop = ["48th Session of the GFCM (November 2025)", "VESSEL"]
        combined = combined[~combined.isin(rows_to_drop).any(axis=1)]

        # Drop specific columns
        cols_to_drop = []
        for col in combined.columns:
            col_values = combined[col].astype(str)
            if (col == "1/201 2/201 3/201 4/201 5/201 6/201 7/201" or
                col == "_1" or
                col == "48th Session of the GFCM (November 2025)" or
                col.startswith("Unknown") or
                col_values.isna().all() or
                (col_values == "").all()):
                cols_to_drop.append(col)

        # Remove first column if it's empty
        if combined.columns[0] == '' or combined.iloc[:, 0].isna().all():
            combined = combined.drop(columns=[combined.columns[0]])

        # Remove column named "_1"
        #if "_1" in combined.columns:
        #  combined = combined.drop(columns=["_1"])


        combined = combined.drop(columns=cols_to_drop)
        combined.to_csv('combined_all_pages_pivoted.csv', index=False)
        print(f"Saved combined data with shape: {combined.shape}")
        return combined

    return None

# Usage
#file_id = "1Z4QQ2CMN7Ae7BALio6OorBDqzJyaWRUk"
#df_gfcm = process_pdf(file_id)
df_gfcm = process_pdf()
'''
Call sign of vessel and previous call sign
Cross-listed with RFMO IUU vessel list
Date of alleged IUU fishing activity
Date vessel was first included on the GFCM IUU List
Flag of vessel
"GFCM IUU vessel list
[November 2025]"
IMO ship identification number
Length overall
MMSI (Maritime Mobile Service Identity) number
"Name and address of owner of vessel and previous owners, including
beneficial owners"
Name of vessel
Operator of vessel and previous operator(s);
Outcome of any action undertaken
Ownersâ€™ place of registration
Photographs of the vessel
Position of alleged IUU fishing activity
Previous flag(s) of vessel
Previous name of vessel
RFMO Reference number
"Summary of activities which justify the inclusion of the vessel on the
list, together with references to all relevant documents informing and
evidencing these activities"
'''
print(df_gfcm.iloc[:, 0].tolist())
df_gfcm = df_gfcm.dropna(how='all')
df_gfcm = df_gfcm.drop(columns=[col for col in df_gfcm.columns if 'GFCM IUU vessel list' in col])

column_names = ['Call Sign', 'Cross-listed', 'Activity Date', 'GFCM IUU Listed Date', 'Flag', 'IMO', 'Length','MMSI','Name Address Owners','Name of vessel','Operators','Outcome','Owner Registration Place','Photograph','Activity Position','Previous Flag','Previous Vessel Name','RFMO Reference Number','Summary Activity']
df_gfcm.columns = column_names

# Create a new DataFrame with mapping
df_map_gfcm = pd.DataFrame({
'Vessel Name': df_gfcm['Name of vessel'],
'IMO Number': df_gfcm['IMO'],
'International Radio Call Sign': df_gfcm['Call Sign'],
'Flag State': df_gfcm['Flag'],
'Vessel Length (LOA, in meters)': df_gfcm['Length'],
'Owner Name': df_gfcm['Name Address Owners'],
'Operator Name': df_gfcm['Operators'],
'Activity Date': df_gfcm['Activity Date'],
'Activity Description': df_gfcm['Summary Activity'],
'Additional Information/Notes': df_gfcm['RFMO Reference Number'],
'Date Listed From': df_gfcm['GFCM IUU Listed Date'],
'RFMO': "GFCM",
'Source File Type': "PDF",
'Source URL': "https://www.fao.org/gfcm/data/iuu-vessel-list/en/",
'Outcome': df_gfcm['Outcome'],
'Previous Names': df_gfcm['Previous Vessel Name'],
'Previous Flag States': df_gfcm['Previous Flag'],
'Cross Listings': df_gfcm['Cross-listed']
})

#print("New columns:", df_gfcm.columns.tolist())
#df_gfcm.to_csv("df_gfcm.csv", index=False)
#files.download("df_gfcm.csv")

df_iuu = pd.concat([df_iuu, df_map_gfcm], ignore_index=True)
#print(f"Number of records: {len(df_iuu)}")




Saved combined data with shape: (374, 20)
[nan, nan, nan, nan, nan, nan, nan, 'CPA 226', 'CPA 202', 'CPA 201', 'CPA 221', 'CPA 222', 'Unknown', 'Unknown', nan, nan, nan, nan, nan, nan, '', '', 'Unknown', '5VAN9', '', '', '', nan, nan, nan, nan, nan, nan, 'PQMG', '', '3CAG', '3CAE', 'Unknown', '', '5IM376', nan, nan, nan, nan, nan, nan, '', '', '', 'FN 003883', 'T8AN3', '', 'HO5381', nan, nan, nan, nan, nan, nan, '', '', '(CPB3000)', '3X07QMC', '', '', '', nan, nan, nan, nan, nan, nan, '', 'JVAW7', '', '', '', '', '', nan, nan, nan, nan, nan, nan, '', '', '', '', '', 'OTS 024 or OTS 089', '', nan, nan, nan, nan, nan, nan, '', '', 'Unknown (4LIN2)', 'HMEK3', '', '', 'Unknown', nan, nan, nan, nan, nan, nan, '', '5NMU', 'Unknown (9LYF36, H03738)', '', 'Unknown', '4SF5385', 'V4BK5 (H3DE)', nan, nan, nan, nan, nan, nan, '(HP7651)', '3ENF8', '', '', '', '', 'H03374 (Unknown)', nan, nan, nan, nan, nan, nan, '', 'Unknown', '', 'Unknown', '', '', '4SF5790', nan, nan, nan, nan, nan, nan, '', 'A4B

In [ ]:
import requests
import pdfplumber
import pandas as pd
import re
from io import BytesIO



# groupby on the cleaned data
def clean_join(series):
    clean_values = series.dropna().astype(str)
    clean_values = clean_values[clean_values.str.strip() != '']
    clean_values = clean_values[clean_values.str.lower() != 'nan']
    clean_values = clean_values.str.replace('-', '')
    clean_values = clean_values.unique()
    return ' '.join(clean_values) if len(clean_values) > 0 else ''

def download_and_extract_seafo_pdf():
    """Download and extract content from SEAFO IUU vessel list PDF"""

    # The URL from your log entry
    pdf_url = "http://www.seafo.org/media/6a4caac4-9965-47b2-b9ad-777f248330aa/SEAFOweb/pdf/IUU/SEAFO%20IUU%20vessel%20list%202026_pdf"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/pdf,application/octet-stream,*/*',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'http://www.seafo.org/',
    }

    print("=== DOWNLOADING SEAFO IUU VESSEL LIST PDF ===")
    print(f"URL: {pdf_url}")

    try:
        # Download the PDF
        response = requests.get(pdf_url, headers=headers, timeout=30)
        print(f"Status Code: {response.status_code}")
        print(f"Content-Type: {response.headers.get('content-type', 'Unknown')}")
        print(f"Content-Length: {len(response.content)} bytes")

        if response.status_code == 200:
            # Save the PDF file
            with open('seafo_iuu_list_2026.pdf', 'wb') as f:
                f.write(response.content)
            print("PDF downloaded successfully!")

            # Extract content using multiple methods
            extract_pdf_content('seafo_iuu_list_2026.pdf')

        else:
            print(f"Failed to download PDF: {response.status_code}")
            print(f"Response: {response.text[:500]}")

    except Exception as e:
        print(f" Error downloading PDF: {e}")

def extract_pdf_content(pdf_path):
    """Extract content from PDF using multiple methods"""

    print(f"\n=== EXTRACTING CONTENT FROM {pdf_path} ===")

    # Using pdfplumber (best for tables)
    try:
        with pdfplumber.open(pdf_path) as pdf:
            print(f"Number of pages: {len(pdf.pages)}")

            all_text = ""
            all_tables = []

            for page_num, page in enumerate(pdf.pages, 1):
                print(f"Processing page {page_num}...")

                # Extract text with better layout preservation
                text = page.extract_text(layout=True, x_tolerance=3, y_tolerance=3)
                if text:
                    all_text += f"\n--- Page {page_num} ---\n{text}\n"

                # Extract tables with settings for complex cells
                tables = page.extract_tables(
                    table_settings={
                        "snap_tolerance": 5,
                        "join_tolerance": 5,
                        "edge_min_length": 3,
                        "min_words_vertical": 1,
                        "text_tolerance": 3,
                        "text_x_tolerance": 3,
                        "text_y_tolerance": 3
                    }
                )

                if tables:
                    for table_num, table in enumerate(tables, 1):
                        all_tables.append({
                            'page': page_num,
                            'table_num': table_num,
                            'data': table
                        })

            # Save extracted text
            with open('seafo_extracted_text.txt', 'w', encoding='utf-8') as f:
                f.write(all_text)
            print("Text saved to seafo_extracted_text.txt")

            # Process tables
            if all_tables:
                process_vessel_tables(all_tables)

    except Exception as e:
        print(f"pdfplumber extraction failed: {e}")


def process_vessel_tables(tables):
    """Process extracted tables to identify vessel information"""

    print(f"\n=== PROCESSING {len(tables)} TABLES FOR VESSEL DATA ===")

    all_vessels = []

    for table_info in tables:
        page = table_info['page']
        table_num = table_info['table_num']
        table_data = table_info['data']

        #print(f"\nProcessing Table {table_num} from Page {page}")

        if not table_data or len(table_data) < 2:
            print("  Skipping: No data or insufficient rows")
            continue

        # Convert to DataFrame for easier processing
        df = pd.DataFrame(table_data[1:], columns=table_data[0])  # First row as headers

        #print(f"  Shape: {df.shape}")
        #print(f"  Columns: {df.columns.tolist()}")

        # Look for vessel-related columns
        vessel_columns = []
        for col in df.columns:
            if col and any(keyword in str(col).lower() for keyword in
                          ['vessel', 'name', 'imo', 'flag', 'call', 'sign', 'owner']):
                vessel_columns.append(col)

        if vessel_columns:
            print(f"  Found vessel columns: {vessel_columns}")

            # Clean and process the data
            df_clean = df.dropna(how='all')  # Remove empty rows

            # Map to standardised format
            vessel_data = []
            for _, row in df_clean.iterrows():
                vessel_info = {
                    'Source_Page': page,
                    'Source_Table': table_num,
                    'RFMO': 'SEAFO',
                    'Source_URL': 'http://www.seafo.org/media/6a4caac4-9965-47b2-b9ad-777f248330aa/SEAFOweb/pdf/IUU/SEAFO%20IUU%20vessel%20list%202026_pdf',
                    'Source_File_Type': 'PDF',
                    'Last_Checked': pd.Timestamp.now().strftime('%Y-%m-%d')
                }

                # Map columns to standard fields
                for col in df.columns:
                    if col:
                        col_lower = str(col).lower()
                        if 'vessel' in col_lower and 'name' in col_lower:
                            vessel_info['Vessel_Name'] = row[col]
                        elif 'imo' in col_lower:
                            vessel_info['IMO_Number'] = row[col]
                        elif 'flag' in col_lower:
                            vessel_info['Flag_State'] = row[col]
                        elif 'call' in col_lower or 'sign' in col_lower:
                            vessel_info['Radio_Call_Sign'] = row[col]
                        elif 'owner' in col_lower:
                            vessel_info['Owner_Name'] = row[col]
                        else:
                            vessel_info[f'Additional_{col}'] = row[col]

                vessel_data.append(vessel_info)

            all_vessels.extend(vessel_data)
            print(f"  Extracted {len(vessel_data)} vessel records")

    if all_vessels:
        # Create final DataFrame
        df_seafo = pd.DataFrame(all_vessels)
        #Forward fill as there are multiple lines per IMO
        df_seafo['IMO_Number'] = df_seafo['IMO_Number'].ffill()

        #df_seafo.to_csv('seafo_iuu_vessels_extracted.csv', index=False)


        #files.download("seafo_iuu_vessels_extracted.csv")


        data_columns = [col for col in df_seafo.columns if col not in ['IMO_Number']]
        df_clean = df_seafo[~df_seafo[data_columns].isna().all(axis=1)].copy()
        other_columns = [col for col in df_clean.columns if col not in ['IMO_Number']]
        agg_dict = {col: clean_join for col in other_columns}
        df_seafo = df_clean.groupby(['IMO_Number']).agg(agg_dict).reset_index()


        # combine duplicate columns for names and operators
        df_seafo.iloc[:, 8] = df_seafo.iloc[:, 8].fillna('').astype(str) + '\n' + df_seafo.iloc[:, 15].fillna('').astype(str)
        df_seafo.iloc[:, 12] = df_seafo.iloc[:, 12].fillna('').astype(str) + '\n' + df_seafo.iloc[:, 16].fillna('').astype(str)
        # Clean up any leading/trailing whitespace and empty lines
        df_seafo.iloc[:, 8] = df_seafo.iloc[:, 8].str.strip().str.replace('^\n+|\n+$', '', regex=True)
        df_seafo.iloc[:, 12] = df_seafo.iloc[:, 12].str.strip().str.replace('^\n+|\n+$', '', regex=True)
        # Drop duplicate columns
        df_seafo = df_seafo.drop(df_seafo.columns[-2:], axis=1)

        # Save to CSV
        #df_seafo.to_csv('seafo_iuu_vessels_extracted.csv', index=False)
        #print(f"\n Extracted {len(all_vessels)} vessel records")
        #print(" Data saved to seafo_iuu_vessels_extracted.csv")


        #files.download("seafo_iuu_vessels_extracted.csv")


        column_names = ['IMO_Number','Source_Page', 'Source_Table', 'RFMO', 'Source_URL','Source_File_Type', 'Last_Checked', 'Vessel_Name',
                        'Previous_Names','Flag_State','IRCS','Summary_Activity','Operators','IUU_Listing_Org','IUU_Listing_Date']
        df_seafo.columns = column_names
        # Create a new DataFrame with mapping
        df_map_seafo = pd.DataFrame({
        'Vessel Name': df_seafo['Vessel_Name'],
        'IMO Number': df_seafo['IMO_Number'],
        'International Radio Call Sign': df_seafo['IRCS'],
        'Flag State': df_seafo['Flag_State'],
        'Operator Name': df_seafo['Operators'],
        'Activity Description': df_seafo['Summary_Activity'],
        'Date Listed From': df_seafo['IUU_Listing_Date'],
        'RFMO': "SEAFO",
        'Source RFMO': df_seafo['IUU_Listing_Org'],
        'Source File Type': "PDF",
        'Source URL': "http://www.seafo.org/media/6a4caac4996547b2b9ad777f248330aa/SEAFOweb/pdf/IUU/SEAFO%20IUU%20vessel%20list%202026_pdf",
        'Last Checked': df_seafo['Last_Checked'],
        'Previous Names': df_seafo['Previous_Names']
        })
        global df_iuu
        df_iuu = pd.concat([df_iuu, df_map_seafo], ignore_index=True)
        print(f"Number of records: {len(df_iuu)}")

        return df_seafo
    else:
        print("\n No vessel data found in tables")
        return None


if __name__ == "__main__":
    # Try the main download first
    download_and_extract_seafo_pdf()

    # If that fails, try direct URLs
    # try_direct_seafo_urls()


=== DOWNLOADING SEAFO IUU VESSEL LIST PDF ===
URL: http://www.seafo.org/media/6a4caac4-9965-47b2-b9ad-777f248330aa/SEAFOweb/pdf/IUU/SEAFO%20IUU%20vessel%20list%202026_pdf
Status Code: 200
Content-Type: application/pdf
Content-Length: 658186 bytes
PDF downloaded successfully!

=== EXTRACTING CONTENT FROM seafo_iuu_list_2026.pdf ===
Number of pages: 11
Processing page 1...
Processing page 2...
Processing page 3...
Processing page 4...
Processing page 5...
Processing page 6...
Processing page 7...
Processing page 8...
Processing page 9...
Processing page 10...
Processing page 11...
Text saved to seafo_extracted_text.txt

=== PROCESSING 23 TABLES FOR VESSEL DATA ===
  Found vessel columns: ['IMO\nNumber', 'Vessel\nName', 'Previous\nNames', 'Current flag and']
  Extracted 23 vessel records
  Found vessel columns: ['IMO\nNumber', 'Vessel\nName']
  Extracted 8 vessel records
  Found vessel columns: ['Current flag and']
  Extracted 16 vessel records
  Found vessel columns: ['IMO\nNumber', 'Ves

In [ ]:
import pandas as pd
import pdfplumber
import requests
from io import BytesIO

def download_and_read_sprfmo_pdf():
    """Download and extract tables from SPRFMO IUU vessel list PDF"""

    url = "https://www.sprfmo.int/assets/Fisheries/IUU-Lists/2026-SPRFMO-IUU-Vessel-List-Final.pdf"

    # Download PDF
    print("Downloading PDF...")
    response = requests.get(url)
    pdf_file = BytesIO(response.content)

    all_tables = []

    with pdfplumber.open(pdf_file) as pdf:
        print(f"PDF has {len(pdf.pages)} pages")

        for page_num, page in enumerate(pdf.pages, 1):
            print(f"Processing page {page_num}...")

            # Extract tables from page
            tables = page.extract_tables()

            if tables:
                for table_num, table in enumerate(tables):
                    if table and len(table) > 1:  # Skip empty tables
                        print(f"  Table {table_num + 1}: {len(table)} rows, {len(table[0])} columns")
                        print(f"  Headers: {table[0]}")

                        # Convert to DataFrame with unique column names
                        headers = table[0]

                        # Make column names unique
                        unique_headers = []
                        header_counts = {}

                        for header in headers:
                            header_str = str(header) if header is not None else 'Unknown'
                            if header_str in header_counts:
                                header_counts[header_str] += 1
                                unique_headers.append(f"{header_str}_{header_counts[header_str]}")
                            else:
                                header_counts[header_str] = 0
                                unique_headers.append(header_str)

                        # Create DataFrame
                        df = pd.DataFrame(table[1:], columns=unique_headers)
                        df['Source_Page'] = page_num
                        df['Source_Table'] = table_num + 1

                        # Only keep tables with substantial data (more than just headers)
                        if len(df) > 0 and not df.iloc[:, 0].isna().all():
                            all_tables.append(df)
                            print(f"    Added table with {len(df)} data rows")
                        else:
                            print(f"    Skipped empty table")

    if all_tables:
        # Combine tables with different columns using outer join
        print(f"\nCombining {len(all_tables)} tables...")

        # Start with first table
        combined_df = all_tables[0].copy()

        # Add other tables one by one
        for i, df in enumerate(all_tables[1:], 1):
            print(f"Adding table {i+1}...")
            # Use concat with sort=False to handle different columns
            combined_df = pd.concat([combined_df, df], ignore_index=True, sort=False)

        print(f"Combined into {len(combined_df)} total rows with {len(combined_df.columns)} columns")
        print(f"Final columns: {list(combined_df.columns)}")

        return combined_df
    else:
        print("No valid tables found in PDF")
        return pd.DataFrame()

def clean_and_pivot_vessel_data(df_sprfmo):
    """Clean and pivot vessel data"""

    if df_sprfmo.empty:
        return df_sprfmo


    #combine column 1 to 2
    df_sprfmo.iloc[:,1] = df_sprfmo.iloc[:,1].combine_first(df_sprfmo.iloc[:,0])
    #combine column 5 to 4
    for i in range(len(df_sprfmo)):
      if df_sprfmo.iloc[i, 3] == '' or pd.isna(df_sprfmo.iloc[i, 3]):
        df_sprfmo.iloc[i, 3] = df_sprfmo.iloc[i, 4]

    #drop rows when column 4 is null or empty
    df_sprfmo = df_sprfmo[df_sprfmo.iloc[:, 3].notna() & (df_sprfmo.iloc[:, 3] != '')]
    #Keep column 2 and 4
    df_sprfmo = df_sprfmo.iloc[:, [1, 3]]

    #Transpose not working due to duplicates
    #df_pivoted = df.set_index(df.columns[0])[df.columns[1]].to_frame().T

    # Replace empty strings with NaN, then forward fill
    df_sprfmo.iloc[:, 0] = df_sprfmo.iloc[:, 0].replace('', pd.NA).ffill()
    # Group and combine
    df_sprfmo = df_sprfmo.groupby(df_sprfmo.columns[0])[df_sprfmo.columns[1]].apply(' '.join).reset_index()

    # Extract vessel name
    vessel_name = df_sprfmo.columns[1].replace('\n', ' ')
    data_dict = dict(zip(df_sprfmo.iloc[:, 0], df_sprfmo.iloc[:, 1]))
    # See what the dictionary looks like
    data_dict = dict(zip(df_sprfmo.iloc[:, 0], df_sprfmo.iloc[:, 1]))
    # Add vessel as the first item
    data_dict = {'Vessel': vessel_name, **data_dict}
    df_sprfmo = pd.DataFrame([data_dict])
    column_names = ['Vessel_Name','Actions_Taken','Activities','Call_Sign','Current_Operator','Flag','IMO','IUU_Vsl_List','Image','Other_id','Outcome','Owner Name and Address','Position','Previous_Names','Prior_Operators','Prior_Owners']
    df_sprfmo.columns = column_names
    df_sprfmo['Owner_Name'] = df_sprfmo['Owner Name and Address'].str.split('\n').str[0].str.strip()
    df_sprfmo['Owner_Address'] = df_sprfmo['Owner Name and Address'].str.split('\n').str[1:].apply(lambda x: ', '.join(line.strip() for line in x if line.strip()))

    # Create a new DataFrame with mapping
    df_map_sprfmo = pd.DataFrame({
    'Vessel Name': df_sprfmo['Vessel_Name'],
    'IMO Number': df_sprfmo['IMO'],
    'International Radio Call Sign': df_sprfmo['Call_Sign'],
    'Flag State': df_sprfmo['Flag'],
    'Owner Name': df_sprfmo['Owner_Name'],
    'Owner Address': df_sprfmo['Owner_Address'],
    'Operator Name': df_sprfmo['Current_Operator'],
    'Activity Description': df_sprfmo['Activities'],
    'Additional Information/Notes': df_sprfmo['Position'],
    'Date Listed From': df_sprfmo['Actions_Taken'],
    'RFMO': "SPRFMO",
    'Source RFMO': df_sprfmo['IUU_Vsl_List'],
    'Source File Type': "PDF",
    'Source URL': "https://www.sprfmo.int/assets/Fisheries/IUU-Lists/2026-SPRFMO-IUU-Vessel-List-Final.pdf",
    'Last Checked': "",
    'IUU Number':df_sprfmo['Other_id'],
    'Action Taken': df_sprfmo['Actions_Taken'],
    'Outcome': df_sprfmo['Outcome'],
    'Previous Names': df_sprfmo['Previous_Names'],
    'Previous Operators': df_sprfmo['Prior_Operators'],
    'Previous Owners': df_sprfmo['Prior_Owners']
    })
    global df_iuu
    df_iuu = pd.concat([df_iuu, df_map_sprfmo], ignore_index=True)
    print(f"Number of records: {len(df_iuu)}")

    return df_sprfmo

# Main execution
if __name__ == "__main__":
    try:
        # Download and extract tables
        df_raw = download_and_read_sprfmo_pdf()

        if not df_raw.empty:
            print(f"\nRaw data shape: {df_raw.shape}")

            # Clean and process the data
            df_processed = clean_and_pivot_vessel_data(df_raw)

            # Save both versions
            #df_raw.to_csv('sprfmo_raw_data.csv', index=False)
            #df_processed.to_csv('sprfmo_processed_data.csv', index=False)
            #files.download('sprfmo_processed_data.csv')

            print("\nFiles saved:")
            print("- sprfmo_raw_data.csv (all extracted data)")
            print("- sprfmo_processed_data.csv (processed vessel data)")

        else:
            print("No data extracted from PDF")

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

PDF has 2 pages
Processing page 1...
  Table 1: 22 rows, 6 columns
  Headers: ['', 'Vessel', '', 'XING FA\n(effective 2025-12)', None, None]
    Added table with 21 data rows
  Table 2: 2 rows, 1 columns
  Headers: ['Prior']
    Added table with 1 data rows
Processing page 2...
  Table 1: 5 rows, 4 columns
  Headers: ['', 'Vessel', '', 'XING FA\n(effective 2025-12)']
    Added table with 4 data rows
  Table 2: 2 rows, 1 columns
  Headers: ['Actions']
    Added table with 1 data rows

Combining 4 tables...
Adding table 2...
Adding table 3...
Adding table 4...
Combined into 27 total rows with 10 columns
Final columns: ['', 'Vessel', '_1', 'XING FA\n(effective 2025-12)', 'Unknown', 'Unknown_1', 'Source_Page', 'Source_Table', 'Prior', 'Actions']

Raw data shape: (27, 10)
Number of records: 1393

Files saved:
- sprfmo_raw_data.csv (all extracted data)
- sprfmo_processed_data.csv (processed vessel data)


In [ ]:
import requests
import pdfplumber
import pandas as pd
from io import BytesIO
from google.colab import files

def scrape_npfc_iuu_list(url: str) -> pd.DataFrame:
    """
    Downloads and scrapes the NPFC IUU Vessel List PDF from the given URL.
    Returns a DataFrame with the extracted data.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    print(f"Downloading PDF from:\n{url}\n")
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    print(f"Download successful. Status code: {response.status_code}")

    pdf_bytes = BytesIO(response.content)
    all_rows = []

    with pdfplumber.open(pdf_bytes) as pdf:
        print(f"PDF has {len(pdf.pages)} page(s).\n")

        for page_num, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables()

            if tables:
                for table in tables:
                    for row in table:
                        # Filter out completely empty rows
                        if any(cell and cell.strip() for cell in row):
                            all_rows.append(row)
            else:
                # Fallback: extract raw text if no tables are detected
                text = page.extract_text()
                if text:
                    print(f"[Page {page_num}] No table found. Raw text:\n{text}\n")

    if not all_rows:
        print("No tabular data found in the PDF.")
        return pd.DataFrame()

    # Use the first row as header if it looks like a header row
    df = pd.DataFrame(all_rows[1:], columns=all_rows[0])

    # Clean up column names and cell values
    df.columns = [str(col).strip() if col else f"col_{i}" for i, col in enumerate(df.columns)]
    df = df.map(lambda x: x.strip().replace("\n", " ") if isinstance(x, str) else x)

    return df


if __name__ == "__main__":
    URL = (
        "https://www.npfc.int/system/files/2025-04/"
        "2025%20NPFC%20IUU%20Vessel%20List_20250327.pdf"
    )

    df = scrape_npfc_iuu_list(URL)

    if not df.empty:
        print(f"Extracted {len(df)} rows and {len(df.columns)} columns.\n")
        print("Preview:")
        print(df.head(10).to_string(index=False))

        output_file = "npfc_iuu_vessel_list.csv"

        # Identify rows where col_0 is a number
        number_mask = pd.to_numeric(df['col_0'], errors='coerce').notna()
        number_indices = df.index[number_mask].tolist()

        # Get the second row after each numbered row
        second_after_indices = [i + 2 for i in number_indices if i + 2 < len(df)]

        # Extract the summary text (column index 1)
        summaries = df.iloc[second_after_indices, 1].values

        # Extract the "Other Information" text from the k. column
        # Based on your data structure, this is column index 4
        other_info = df.iloc[second_after_indices, 4].values

        # Filter to keep only the numbered rows
        df_npfc = df.iloc[number_indices].copy()
        #print(df_npfc.columns.tolist())
        column_names = ['No','Name_of_Vessel','Flag_of_Vessel','Owner','Operator','Call_Sign','Lloyds/IMO','Date_First_Included','CMM_Paragraph']
        df_npfc.columns = column_names

        # Insert both as new columns
        df_npfc['Summary'] = summaries
        df_npfc['Other_Information'] = other_info

         # Create a new DataFrame with mapping
        df_map_npfc = pd.DataFrame({
        'Vessel Name': df_npfc['Name_of_Vessel'],
        'IMO Number': df_npfc['Lloyds/IMO'],
        'International Radio Call Sign': df_npfc['Call_Sign'],
        'Flag State': df_npfc['Flag_of_Vessel'],
        'Owner Name': df_npfc['Owner'],
        'Operator Name': df_npfc['Operator'],
        'Additional Information/Notes': df_npfc['Other_Information'].fillna('') + " " + df_npfc['CMM_Paragraph'].fillna(''),
        'Date Listed From': df_npfc['Date_First_Included'],
        'RFMO': "NPFC",
        'Source File Type': "PDF",
        'Source URL': "https://www.npfc.int/npfc-iuu-vessel-list",
        })

        df_iuu = pd.concat([df_iuu, df_map_npfc], ignore_index=True)
        print(f"Number of records: {len(df_iuu)}")

        #df_npfc.to_csv(output_file, index=False, encoding="utf-8-sig")
        #files.download(output_file)
        #print(f"\nData saved to: {output_file}")
    else:
        print("No data extracted.")

https://www.npfc.int/system/files/2025-04/2025%20NPFC%20IUU%20Vessel%20List_20250327.pdf

Download successful. Status code: 200
PDF has 40 page(s).

Extracted 326 rows and 9 columns.

Preview:
col_0                                                                                                                                                                                                                                                                                                                                                                              a. Name of vessel b. Flag of vessel  c. Owner   d. Operator of                           e. Call sign of f. Lloyds/IMO h. Date first included on i. CMM and paragraph
  No.                                                                                                                                                                                                                                                                         

In [ ]:
import requests
import pdfplumber
import pandas as pd
import io

# --- Configuration ---
PDF_URL = "https://siofa.org/sites/default/files/SIOFA-IUU-Vessels-List-2025-10-09.pdf"
START_PAGE = 2  # 0-indexed, so page 3 = index 2
OUTPUT_FILE = "siofa_iuu_vessels.csv"

# --- Download PDF ---
print(f"Downloading PDF from:\n{PDF_URL}\n")
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(PDF_URL, headers=headers)
print(f"Download {'successful' if response.status_code == 200 else 'failed'}. Status code: {response.status_code}")

# --- Extract Tables ---
all_rows = []

with pdfplumber.open(io.BytesIO(response.content)) as pdf:
    print(f"PDF has {len(pdf.pages)} page(s).")

    for page in pdf.pages[START_PAGE:]:
        table = page.extract_table()
        if table:
            all_rows.extend(table)

# --- Build DataFrame ---
# Use first row as header
df_siofa = pd.DataFrame(all_rows[1:], columns=all_rows[0])

# Drop empty rows
df_siofa = df.dropna(how='all')
df_siofa = df_siofa[df_siofa.iloc[:, 0].str.strip().str.match(r'^\d+$', na=False)]

# Clean whitespace
df_siofa = df_siofa.map(lambda x: x.strip().replace("\n", " ") if isinstance(x, str) else x)

print(f"\nExtracted {len(df_siofa)} vessel records.")
print("\nPreview:")
print(df.head(10).to_string())

# --- Save to CSV ---
df_siofa.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"\nData saved to: {OUTPUT_FILE}")
print(df_siofa.columns.tolist())
# ['col_0', 'a. Name of vessel', 'b. Flag of vessel', 'c. Owner', 'd. Operator of', 'e. Call sign of', 'f. Lloyds/IMO', 'h. Date first included on', 'i. CMM and paragraph']
column_names = ['No','Name','Flag of vessel','Owner','Operator of','Call sign of','Lloyds/IMO','Date first included on','CMM and paragraph']
df_siofa.columns = column_names
# Create a new DataFrame with mapping
df_map_siofa = pd.DataFrame({
'Vessel Name': df_siofa['Name'],
'IMO Number': df_siofa['Lloyds/IMO'],
'International Radio Call Sign': df_siofa['Call sign of'],
'Flag State': df_siofa['Flag of vessel'],
'Owner Name': df_siofa['Owner'],
'Operator Name': df_siofa['Operator of'],
'Additional Information/Notes': df_siofa['CMM and paragraph'],
'Date Listed From': df_siofa['Date first included on'],
'RFMO': "SIOFA",
'Source File Type': "PDF",
'Source URL': "https://siofa.org/sites/default/files/SIOFA-IUU-Vessels-List-2025-10-09.pdf",
})

df_iuu = pd.concat([df_iuu, df_map_siofa], ignore_index=True)
print(f"Number of records: {len(df_iuu)}")

# Uncomment if running in Google Colab
#from google.colab import files
#files.download(OUTPUT_FILE)

https://siofa.org/sites/default/files/SIOFA-IUU-Vessels-List-2025-10-09.pdf

Download successful. Status code: 200
PDF has 26 page(s).

Extracted 40 vessel records.

Preview:
  col_0                                                                                                                                                                                                                                                                                                                                                                               a. Name of vessel  b. Flag of vessel   c. Owner    d. Operator of                            e. Call sign of  f. Lloyds/IMO  h. Date first included on  i. CMM and paragraph
0   No.                                                                                                                                                                                                                                                                               

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://www.ccamlr.org/en/compliance/iuu-vessel-lists"
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def fetch_table(url):
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Replace <br> and <li> tags with newline characters
    for tag in soup.find_all(['br', 'li']):
        tag.insert_before('\n')

    tables = soup.find_all("table")

    if not tables:
        print("No tables found.")
        return None

    all_dataframes = []

    for table in tables:
        headers = [th.get_text(strip=True) for th in table.find_all("th")]

        rows = []
        for tr in table.find_all("tr")[1:]:
            cells = [td.get_text(strip=False).strip() for td in tr.find_all("td")]
            if cells:
                rows.append(cells)

        if rows:
            df = pd.DataFrame(rows, columns=headers if headers else None)
            all_dataframes.append(df)

    return all_dataframes


if __name__ == "__main__":
    dfs = fetch_table(URL)

    #if dfs:
    #    for i, df in enumerate(dfs):
    #        print(f"\n=== Table {i+1} ===")
    #        print(df.head())
    df_ccamlr = dfs[0]

    # Create a new DataFrame with mapping
    # Vessel name	Flag	IMO Number	Callsign	Nature and date of activity(s)	Year Listed	Ownership history
    df_map_ccamlr = pd.DataFrame({
    'Vessel Name': df_ccamlr['Vessel name'],
    'IMO Number': df_ccamlr['IMO Number'],
    'International Radio Call Sign': df_ccamlr['Callsign'],
    'Flag State': df_ccamlr['Flag'],
    'Owner Name': df_ccamlr['Ownership history'],
    'Activity Description': df_ccamlr['Nature and date of activity(s)'],
    'Date Listed From': df_ccamlr['Year Listed'],
    'RFMO': "CCAMLR",
    'Source File Type': "HTML",
    'Source URL': "https://www.ccamlr.org/en/compliance/iuu-vessel-lists",
    })
    #Previous Owners:

    df_iuu = pd.concat([df_iuu, df_map_ccamlr], ignore_index=True)
    print(f"Number of records: {len(df_iuu)}")
    # save first table
    #dfs[0].to_csv("ccamlr_iuu_vessels.csv", index=False, encoding='utf-8-sig')
    #print("\nSaved to ccamlr_iuu_vessels.csv")

    #from google.colab import files
    #files.download("ccamlr_iuu_vessels.csv")

Number of records: 1491


In [ ]:
df_iuu.to_csv("iuu_vessels.csv", index=False)
from google.colab import files
files.download("iuu_vessels.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
import io
from google.colab import userdata
import pandas as pd

# Fake dataframe
df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie", "Diana"],
    "Age": [25, 30, 35, 28],
    "City": ["Singapore", "London", "New York", "Sydney"],
    "Salary": [5000, 6000, 7000, 5500]
})

sender = "linernest@gmail.com"
receiver = "ernest_lin@sfa.gov.sg"
password = userdata.get('EMAIL')

msg = MIMEMultipart()
msg["From"] = sender
msg["To"] = receiver
msg["Subject"] = "Hello from Colab"
msg.attach(MIMEText("Please find the attached CSV file.", "plain"))

# Convert dataframe to CSV in memory
csv_buffer = io.StringIO()
df.to_csv(csv_buffer, index=False)
csv_bytes = csv_buffer.getvalue().encode()

# Attach it
attachment = MIMEBase("application", "octet-stream")
attachment.set_payload(csv_bytes)
encoders.encode_base64(attachment)
attachment.add_header("Content-Disposition", "attachment; filename=data.csv")
msg.attach(attachment)

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
    server.login(sender, password)
    server.sendmail(sender, receiver, msg.as_string())

print("Email sent successfully!")

Email sent successfully!
